# Phase 3b — improved RAG (Arm B2)

Thin Colab runner for **Arm B2**: one round of retrieval improvements over the naive RAG of Arm B, chosen **only on API recall** (never on pass@1):

- corpus v2 (SQL reference pages, pre-1.0 upgrade guides, `Series.*` twins of `Expr.*` pages and tiny chunks without examples dropped);
- variants a = v2 + granite, b = v2 + Qwen3-Embedding-0.6B, c = the better of a/b + draft-code query (instruction + the model's own zero-shot draft, i.e. the Arm A answer);
- grid k in {5, 8} x context budget in {2000, 3000, 4000}; the best config by non-core API recall is used for generation.

This notebook only clones the repo, installs pinned dependencies and calls scripts. All logic lives in the repo. **Runtime:** GPU, target **L4 (24 GB)**.

In [ ]:
# Confirm the GPU (expect an L4 or A100).
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the repo (idempotent) and enter it.
import os
REPO = 'https://github.com/yusuftanriverdi24/Polars-Rag---Finetune.git'
if not os.path.isdir('polars-rag-vs-ft'):
    !git clone -q $REPO polars-rag-vs-ft
%cd polars-rag-vs-ft
!git pull -q

In [ ]:
# Install pinned dependencies: `colab` = GPU stack, `rag` = sentence-transformers +
# LanceDB (exact pins). torch stays Colab's preinstalled CUDA build.
!pip install -q -e ".[colab,rag]"

In [ ]:
# 1. Download the Polars 2.0.0 docs corpus at the pinned commit -> data/raw/.
!python -m polars_llm.rag.ingest

In [ ]:
# 2. Corpus v1 (the Arm B corpus) -> data/rag/chunks.jsonl; it must be identical to the one Arm B used.
!python -m polars_llm.rag.chunk --stats /tmp/chunk_stats_v1.json
!python -c "import json; a=json.load(open('results/B/generation_meta.json'))['rag']['index_meta']['chunks_sha256']; b=json.load(open('/tmp/chunk_stats_v1.json'))['chunks_sha256']; print('corpus v1 identical to Arm B:', a==b)"

In [ ]:
# 3. Corpus v2 = v1 minus the noisy chunk families -> data/rag/chunks_v2.jsonl (before/after counts printed).
!python -m polars_llm.rag.corpus

In [ ]:
# 4. Indexes: v1 + granite (the Arm B setup, reference row only), v2 + granite (a), v2 + Qwen3-Embedding (b).
!python -m polars_llm.rag.index --device cuda --batch-size 32
!python -m polars_llm.rag.index --chunks data/rag/chunks_v2.jsonl --index-dir data/rag/index_v2_granite --embedder granite --device cuda --batch-size 32
!python -m polars_llm.rag.index --chunks data/rag/chunks_v2.jsonl --index-dir data/rag/index_v2_qwen3 --embedder qwen3 --device cuda --batch-size 16

In [ ]:
# 5. Retrieval comparison (API recall only): naive / a / b / c x k {5, 8} x budget {2000, 3000, 4000}.
#    Picks the B2 config by non-core recall and the max_seq_length it needs
#    -> results/rag/retrieval_eval_v2.json.
!python -m polars_llm.rag.eval_retrieval_v2 --device cuda

In [ ]:
# 6. Generate Arm B2 with the chosen config (index, query mode, k, budget, max_seq_length).
#    Draft-code query: the drafts are the committed Arm A generations (same model + prompt, checked).
!python scripts/generate.py --arm B2 --model Qwen/Qwen3.5-4B --batch-size 8 --retrieval-config results/rag/retrieval_eval_v2.json

In [ ]:
# 7. Score on CPU (sandbox execution under Polars 2.0.0).
!python scripts/score.py --arm B2

In [ ]:
# 8. Same determinism check every arm gets: re-run order-sensitive generations 50x.
!python benchmark/determinism_audit.py --scores results/B2/scores.json --out results/B2/determinism_audit.json | tail -n 25

In [ ]:
# 9. Summary: metrics + end-to-end cost (draft + retrieval + generation).
!python -c "import json; m=json.load(open('results/B2/scores.json'))['metrics']; print('pass@1', m['pass'], '/', m['total'], '=', m['pass_at_1']); print(m['status_counts']); print('by category', {k: v['pass_at_1'] for k, v in m['pass_at_1_by_category'].items()})"
!python -c "import json; g=json.load(open('results/B2/generation_meta.json')); print('config', {k: g['rag'][k] for k in ('query_mode', 'k', 'context_budget_tokens', 'index_dir')}); print('max_seq_length', g['max_seq_length'], 'max prompt', g['max_prompt_tokens']); print('cost', g['cost'])"

In [ ]:
# 10. Zip results for download.
!zip -q -r results_B2.zip results/B2 results/rag/corpus_v2_stats.json results/rag/retrieval_eval_v2.json
from google.colab import files
files.download('results_B2.zip')